# 02 · Feature Engineering
**Flight Analytics Platform**

This notebook demonstrates:
- Deriving velocity, altitude, and heading features
- Classifying flights into altitude bands
- Geographic region classification
- Correlation analysis for ML feature selection

In [ ]:
import os, warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import psycopg2
import plotly.express as px
import plotly.graph_objects as go

def get_conn():
    return psycopg2.connect(
        dbname  = os.getenv('POSTGRES_DB',       'flightdw'),
        user    = os.getenv('POSTGRES_USER',     'flightuser'),
        password= os.getenv('POSTGRES_PASSWORD', 'flightpass123'),
        host    = os.getenv('POSTGRES_HOST',     'postgres'),
        port    = os.getenv('POSTGRES_PORT',     '5432'),
    )

def sql(q):
    with get_conn() as c: return pd.read_sql(q, c)

print('✅ Connected')

## 1. Load Raw Data

In [ ]:
df = sql("""
    SELECT icao24, callsign, origin_country,
           longitude, latitude, baro_altitude, geo_altitude,
           velocity, velocity_kmh, true_track, vertical_rate,
           on_ground, altitude_level, flight_region, ingested_at
    FROM staging.flight_events_clean
    WHERE ingested_at >= NOW() - INTERVAL '2 hours'
    LIMIT 10000
""")
print(f'Rows: {len(df):,}  Columns: {df.shape[1]}')
df.dtypes

## 2. Velocity Transformations

In [ ]:
df['velocity_kmh_recalc'] = df['velocity'] * 3.6
df['velocity_log']        = np.log1p(df['velocity_kmh'].clip(lower=0))
df['velocity_bin']        = pd.cut(
    df['velocity_kmh'],
    bins=[0, 150, 350, 600, 900, 1500],
    labels=['slow', 'medium', 'fast', 'very_fast', 'supersonic']
)

print(df[['velocity','velocity_kmh','velocity_log','velocity_bin']].head(10))

In [ ]:
fig = go.Figure()
bins = np.arange(0, 1200, 30)
v    = df['velocity_kmh'].dropna()

fig.add_trace(go.Histogram(x=v, xbins=dict(start=0, end=1200, size=30),
                           name='Velocity km/h', marker_color='#00d4ff', opacity=0.8))
fig.update_layout(title='Velocity Distribution (km/h)', template='plotly_dark',
                  xaxis_title='Speed (km/h)', yaxis_title='Count')
fig.show()

## 3. Altitude Feature Engineering

In [ ]:
def classify_altitude(alt):
    if pd.isna(alt): return 'unknown'
    if alt <= 0:    return 'ground'
    if alt < 1000:  return 'low'
    if alt < 6000:  return 'medium'
    if alt < 12000: return 'high'
    return 'very_high'

df['altitude_class']   = df['baro_altitude'].apply(classify_altitude)
df['altitude_log']     = np.log1p(df['baro_altitude'].clip(lower=0))
df['altitude_geo_diff']= (df['geo_altitude'].fillna(0) - df['baro_altitude'].fillna(0)).abs()

print(df[['baro_altitude','altitude_class','altitude_log']].head(8))

In [ ]:
alt_counts = df['altitude_class'].value_counts().reset_index()
alt_counts.columns = ['level','count']

colour_map = {
    'ground':'#ef4444','low':'#f97316','medium':'#eab308',
    'high':'#22c55e','very_high':'#3b82f6','unknown':'#6b7280'
}
fig = px.bar(alt_counts, x='level', y='count',
             color='level', color_discrete_map=colour_map,
             title='Altitude Band Frequency', template='plotly_dark')
fig.update_layout(showlegend=False)
fig.show()

## 4. Heading / True Track Analysis

In [ ]:
df['heading_rad']  = np.deg2rad(df['true_track'].fillna(0))
df['heading_sin']  = np.sin(df['heading_rad'])
df['heading_cos']  = np.cos(df['heading_rad'])

# Circular histogram of headings
headings = df['true_track'].dropna()
fig = go.Figure(go.Barpolar(
    r    = np.histogram(headings, bins=36, range=(0,360))[0],
    theta= list(range(0, 360, 10)),
    width= 10,
    marker_color='#7c3aed',
    opacity=0.9,
))
fig.update_layout(
    title='True Track (Heading) Distribution',
    template='plotly_dark',
    polar=dict(bgcolor='#0d1117',
               radialaxis=dict(gridcolor='#333'),
               angularaxis=dict(direction='clockwise', gridcolor='#333')),
)
fig.show()

## 5. Correlation Heatmap

In [ ]:
num_cols = ['baro_altitude','velocity_kmh','true_track','vertical_rate',
            'altitude_log','velocity_log','heading_sin','heading_cos']
corr = df[num_cols].corr()

fig = px.imshow(
    corr, text_auto='.2f',
    color_continuous_scale='RdBu_r',
    title='Feature Correlation Matrix',
    template='plotly_dark',
)
fig.update_layout(height=500)
fig.show()

## 6. On-Ground vs Airborne Statistics

In [ ]:
df['status'] = df['on_ground'].map({True:'Ground', False:'Airborne'}).fillna('Unknown')

fig = px.box(
    df[df['velocity_kmh'].between(0, 1200)],
    x='status', y='velocity_kmh', color='status',
    title='Speed Distribution: Ground vs Airborne',
    template='plotly_dark',
    labels={'velocity_kmh': 'Speed (km/h)'},
)
fig.show()

## 7. Final Feature Summary

In [ ]:
feature_cols = ['velocity_kmh','velocity_log','baro_altitude','altitude_log',
                'vertical_rate','true_track','heading_sin','heading_cos']
summary = df[feature_cols].agg(['count','mean','std','min','max']).round(3)
display(summary)